# Clase 3: Claude Code

<h2 style="color: #e67e22;">El bucle agéntico</h2>

##### Definición de Agente:
Un agente es una entidad que percibe su entorno a través de sensores y actúa sobre ese entorno a través de actuadores. Formalmente, el comportamiento de un agente se describe mediante la función del agente, que mapea cualquier secuencia de percepciones dada a una acción.  

Cuando le da una tarea a Claude, trabaja a través de tres fases: recopilar contexto, tomar acción y verificar resultados. Estas fases se mezclan entre sí. Claude utiliza herramientas en todo momento, ya sea buscando archivos para entender su código, editando para hacer cambios o ejecutando pruebas para verificar su trabajo.
Diagrama del bucle agentico: Su indicación lleva a Claude a recopilar contexto, tomar acción, verificar resultados y repetir hasta completar la tarea. Puede interrumpir en cualquier momento.

![Diagrama Pipeline LLM](img/agentic-loop.svg)



##### Subagentes integrados.  
Claude Code incluye subagentes integrados que Claude utiliza automáticamente cuando es apropiado. Cada uno hereda los permisos de la conversación principal con restricciones de herramientas adicionales.

[anthropic-docs]: https://code.claude.com/docs/es/sub-agents#explore
Consulta la [documentación][anthropic-docs].



# 1. CLAUDE.md — Jerarquía y Organización

<h2 style="color: #e67e22;">Tres niveles de configuración</h2>

Claude Code carga CLAUDE.md desde tres ubicaciones, en orden de prioridad creciente:

```text
1. ~/.claude/CLAUDE.md                          ← user-level (no compartido con el equipo)
2. .claude/CLAUDE.md o CLAUDE.md                ← project-level (versionado)
3. subdirectorio/CLAUDE.md                      ← directory-level (aplica solo bajo ese path)  
```



---

<h2 style="color: #e67e22;">Sintaxis @import: modularidad</h2>

Para mantener CLAUDE.md modular, se usa @import para incluir archivos externos:

# CLAUDE.md (raíz del proyecto)

@import .claude/rules/coding-standards.md  
@import .claude/rules/testing.md   
@import packages/api/standards.md  
  
Esto permite separar reglas por dominio o por mantenedor, e incluir solo las relevantes en cada paquete de un monorepo.

---

<h2 style="color: #e67e22;">El directorio .claude/rules/</h2>

Una alternativa al CLAUDE.md monolítico es .claude/rules/, donde cada archivo agrupa reglas por tema:   

```text
.claude/rules/  
├── testing.md  
├── api-conventions.md  
├── deployment.md  
└── git-workflow.md  
```  

```text
---
paths:
  - "src/api/**/*.ts"
---

# API Development Rules

- All API endpoints must include input validation
- Use the standard error response format
- Include OpenAPI documentation comments
```  

Las reglas sin campo paths se cargan incondicionalmente y aplican a todos los archivos. Las reglas scopeadas por path se disparan cuando Claude lee archivos que coinciden con el patrón, no en cada uso de herramienta.

---

<h2 style="color: #e67e22;">El comando /memory</h2>

Para diagnosticar comportamiento inconsistente entre sesiones, el comando `/memory` muestra exactamente qué archivos de memoria está cargando Claude Code. 

**Útil cuando:**
- Un miembro del equipo no recibe ciertas instrucciones.
- El comportamiento difiere entre directorios del mismo proyecto.
- Sospechás que un user-level CLAUDE.md está sobrescribiendo el del proyecto.

---

[anthropic-docs]: https://code.claude.com/docs/es/sub-agents#explore
Consulta la [documentación][anthropic-docs].


# 2. Commands, Skills

<h2 style="color: #e67e22;">Commands: workflows reutilizables</h2>  

Los commands son prompts pre-definidos que se invocan con `/` en Claude Code. Permiten estandarizar workflows que el equipo ejecuta frecuentemente.

#### Project-scoped: `.claude/commands/`  

Compartidos via version control. Todo el equipo tiene acceso:

```text
.claude/  
└── commands/  
    ├── review-pr.md       # /review-pr  
    ├── add-test.md        # /add-test  
    └── deploy-check.md    # /deploy-check
```

Se invocan como `/review-pr` en la conversación.

#### User-scoped: `~/.claude/commands/`  

Personales, no compartidos. Para workflows individuales:   

```text
~/.claude/  
└── commands/  
    ├── daily-standup.md   # /daily-standup  
    └── my-review.md       # /my-review  
```

#### Cuándo crear un command
- El equipo ejecuta el mismo workflow repetidamente
- El prompt requiere instrucciones específicas que se olvidan
- Quieres estandarizar cómo el equipo usa Claude para una tarea


[anthropic-docs]: https://code.claude.com/docs/es/commands#comandos
Consulta la [documentación][anthropic-docs].
  

---

<h2 style="color: #e67e22;">Skills: capacidades on-demand</h2>  

Las skills son más poderosas que los commands. Viven en `.claude/skills/` con un archivo SKILL.md que define comportamiento, restricciones y metadatos.  

**Estructura básica**  
```text
.claude/  
└── skills/  
    ├── sdd-explore/  
    │   └── SKILL.md  
    ├── create-api-endpoint/  
    │   └── SKILL.md  
    └── migrate-database/  
        └── SKILL.md  
```


---

<h2 style="color: #e67e22;">SKILL.md</h2>  

### Con frontmatter

```text
---  
context: fork
allowed-tools: ["Read", "Grep", "Glob"]
argument-hint: "nombre del módulo a explorar"
---
# Explorar módulo
Analiza la estructura del módulo especificado:
1. Identifica archivos principales con Glob
2. Lee los entry points con Read
3. Busca dependencias con Grep
4. Genera un resumen de arquitectura
## Output esperado
- Archivos principales y su propósito
- Dependencias internas y externas
- Patrones de diseño identificados
```

### Frontmatter: context

context: `fork`  

`fork`: Ejecuta la skill en un contexto aislado (fork del conversation). El output verboso de la exploración no contamina la conversación principal. Solo el resultado final se retorna al main thread.  

**Sin context (default):** la skill se ejecuta inline en la conversación actual. Apropiado para skills cortas que no generan mucho output intermedio.  

**Cuándo usar fork:**  
- Skills de exploración que leen muchos archivos
- Análisis extensos que generan output intermedio largo
- Cuando quieres que solo el resumen final aparezca en la conversación

### Frontmatter: allowed-tools

**allowed-tools:** `["Read", "Grep", "Glob", "Bash"]`  

Restringe qué tools puede usar Claude durante la ejecución de la skill. Esto es una medida de **seguridad y enfoque**:
- Una skill de solo lectura no necesita Write ni Edit
- Una skill de análisis no necesita Bash
- Restringir tools previene side effects no deseados
- Si no se especifica, la skill tiene acceso a todos los tools disponibles.


### Frontmatter: argument-hint
**argument-hint:** `"ruta del archivo o módulo a analizar"`  

Cuando un usuario invoca la skill sin argumentos, Claude usa este hint para solicitar los parámetros necesarios.  

Mejora la UX al guiar al usuario:
```text
> /explore-module
Claude: ¿Cuál es la ruta del archivo o módulo a analizar?
> src/services/auth
```

---

| Característica       | Cuándo se carga                                              | Qué se carga                                              | Costo de contexto                                          |
| ------------------- | ----------------------------------------------------------- | -------------------------------------------------------- | ---------------------------------------------------------- |
| CLAUDE.md           | Inicio de sesión                                            | Contenido completo                                       | Cada solicitud                                             |
| Rules (.claude/rules/) | Inicio de sesión (sin `paths`); al leer archivos que coinciden (con `paths`) | Contenido completo del archivo de regla                  | Cada solicitud si no tiene `paths`; bajo/diferido si está scopeada por path |
| Skills              | Inicio de sesión + cuando se usa                            | Descripciones al inicio, contenido completo cuando se usa | Bajo (descripciones cada solicitud)*                       |
| Servidores MCP      | Inicio de sesión                                            | Todas las definiciones de herramientas y esquemas        | Cada solicitud†                                            |
| Subagents           | Cuando se generan                                           | Contexto fresco con skills especificadas                 | Aislado de la sesión principal                             |
| Hooks               | Al desencadenar                                             | Nada (se ejecuta externamente)                           | Cero, a menos que el hook devuelva contexto adicional      |

*Las descripciones de skills se cargan al inicio para que Claude decida cuándo usarlas. Con `disable-model-invocation: true` el costo baja a cero hasta que la invoques manualmente.
†En la práctica, con *tool search* (activo por defecto), MCP carga herramientas hasta ~10% del contexto y difiere el resto hasta que se usan.

# 3. Plan Mode y Refinamiento Iterativo

<h2 style="color: #e67e22;">Plan Mode vs Direct Execution</h2>  

No todo requiere planificación. La clave es saber cuándo cada approach es apropiado.

### Cuándo usar Plan Mode

| Situación | Por qué planificar |
| :--- | :--- |
| Tareas complejas | Múltiples pasos interdependientes |
| Large-scale changes | Afectan muchos archivos o módulos |
| Decisiones arquitectónicas | Trade-offs que necesitan evaluación |
| Multi-file refactoring | Riesgo de inconsistencias entre archivos |

### Cuándo usar Direct Execution

| Situación | Por qué ejecutar directo |
| :--- | :--- |
| Cambios simples | Un solo archivo, intención clara |
| Bug fixes bien definidos | Se sabe exactamente qué corregir |
| Single-file changes | Sin dependencias cross-file |
| Tareas mecánicas | Renaming, formatting, imports |


  
**Plan Mode permite exploración segura**. La ventaja clave del plan mode es que Claude explora sin commitear cambios. Puede leer archivos, analizar dependencias, y proponer una estrategia antes de tocar código. Si el plan no convence, se ajusta sin haber modificado nada.


---

<h2 style="color: #e67e22;">Tips</h2>  

### 1. Explore Subagent
Para tareas de investigación extensas, usa un explore subagent que aísle el discovery verboso:

```text
<!-- .claude/skills/explore/SKILL.md -->
---
context: fork
allowed-tools: ["Read", "Grep", "Glob"]
argument-hint: "tema o área del codebase a explorar"
---
# Explorar codebase
1. Usa Grep para encontrar archivos relevantes
2. Lee los archivos principales
3. Traza flujos de datos y dependencias
4. Retorna un resumen estructurado con hallazgos
```

Con context: `fork`, toda la lectura de archivos y búsquedas queda aislada. Solo el resumen final llega a la conversación principal.  

---

### 2. Combinar plan + direct execution
El workflow más efectivo para tareas medianas:

```text
1. Plan mode → investigación y diseño
   "Analiza cómo está implementado el sistema de caché actual"
   → Claude explora, lee archivos, propone plan
2. Direct execution → implementación
   "Implementa el paso 1 del plan: crear la interfaz CacheProvider"
   → Claude ejecuta con cambios concretos
```

Esto separa el pensar del hacer, reduciendo errores.

---

### 3. Comunicar transformaciones efectivamente
Concrete input/output > prose descriptions en lugar de describir la transformación en prosa:

❌ "Transforma los datos del usuario para que el nombre esté capitalizado y el email normalizado"  

Proporciona ejemplos concretos de input y output:   

✅ Input:   
   { "name": "juan pérez", "email": "[email protected]" }  
   Output esperado:  
   { "name": "Juan Pérez", "email": "[email protected]" }  

Los ejemplos concretos eliminan ambigüedad. Claude entiende exactamente qué transformación aplicar sin interpretar prosa que puede ser imprecisa.

---

### 4. Test-Driven Iteration  
Escribir tests primero, iterar compartiendo failures.

**Por qué funciona**
- Los tests definen el comportamiento esperado de forma inequívoca
- Los failures dan a Claude información precisa sobre qué falta
- Cada iteración es un ciclo corto y enfocado
- El resultado final tiene cobertura garantizada

---

### 5. Interview Pattern
En lugar de dar todas las instrucciones upfront, deja que Claude pregunte:   

```text
Usuario: "Necesito implementar un sistema de notificaciones"
Claude: "Antes de implementar, necesito entender algunos aspectos:
1. ¿Qué canales? (email, push, SMS, in-app)
2. ¿Las notificaciones son en tiempo real o batch?
3. ¿Hay templates predefinidos o contenido dinámico?
4. ¿Necesitas retry para fallos de delivery?
5. ¿Hay preferencias de usuario (opt-out por canal)?"
```

---

### 6. Batching de issues

##### Todos los issues en un solo mensaje
Apropiado cuando los issues interactúan entre sí:  

```text
"Corrige estos 3 issues en el componente UserForm:
1. El validation message no se limpia al corregir el campo
2. El botón submit se habilita antes de que todos los campos sean válidos
3. El estado del form no se resetea después de submit exitoso"
→ Claude ve las 3 correcciones juntas y puede implementar
   una solución coherente que no cree conflictos entre fixes.
```

##### Issues secuenciales (uno a uno)
Apropiado cuando los issues son independientes:

```text
Mensaje 1: "Agrega lazy loading a las imágenes del catálogo"
→ Claude implementa, tú verificas
Mensaje 2: "Cambia el formato de fecha en el dashboard a DD/MM/YYYY"
→ Claude implementa, tú verificas
```



[anthropic-docs]: https://code.claude.com/docs/es/how-claude-code-works
Consulta la [documentación][anthropic-docs].

# 4. Claude Code en CI/CD

<h2 style="color: #e67e22;">El flag -p (—print) para pipelines</h2>  

Claude Code puede ejecutarse en modo no-interactivo con el flag -p (alias --print):  

 -p "Revisa este archivo y lista los issues de seguridad" < src/auth.ts

**En este modo:**  
- No espera input del usuario (esencial para CI/CD)
- Lee el prompt, ejecuta, y retorna el resultado
- Termina automáticamente al completar
- Compatible con pipes y redirección

**Ejemplo en GitHub Actions**

```yaml
name: Code Review
run: |
  claude -p "Revisa los cambios del PR actual. \
  Enfócate en seguridad, performance y adherencia a convenciones. \
  Lista findings por severidad." > review-output.txt
```  
**Ejemplo en GitLab CI**  

```yaml
review:
  script:
    - claude -p "Analiza los archivos modificados en este MR" > review.txt
    - cat review.txt
```

---

<h2 style="color: #e67e22;">CLAUDE.md como contexto para CI</h2>  

El CLAUDE.md del proyecto es el mecanismo principal para dar contexto a Claude Code en CI/CD. Las instrucciones que Claude sigue localmente también aplican en el pipeline:

```text
<!-- CLAUDE.md -->
## Testing Standards (CI)
- Framework: Vitest
- Coverage mínimo: 80%
- Fixtures en tests/fixtures/
- Mocks en tests/__mocks__/
## Review Criteria
- No console.log en código de producción
- Imports ordenados: built-in → external → internal
- Funciones async deben tener error handling
- No secrets hardcodeados
```

Cuando Claude Code ejecuta en CI con -p, lee CLAUDE.md y aplica estos criterios automáticamente.

---

<h2 style="color: #e67e22;">Structured Output con —output-format</h2>  

Para parsear la salida de Claude en pipelines, usa `--output-format json`:

```text
 -p "Lista los endpoints sin autenticación" \
  --output-format json
```

**Ejemplo:**  
```text
-p "Analiza la cobertura de tests" \
  --output-format json \
  --json-schema '{
    "type": "object",
    "properties": {
      "coverage_percentage": { "type": "number" },
      "uncovered_files": {
        "type": "array",
        "items": { "type": "string" }
      },
      "recommendation": { "type": "string" }
    },
    "required": ["coverage_percentage", "uncovered_files"]
  }'
```

[anthropic-docs]: https://code.claude.com/docs/es/gitlab-ci-cd#claude-code-gitlab-ci-cd
Consulta la [documentación][anthropic-docs].